In [1]:
import openpyxl
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.styles import Font, PatternFill
import pandas as pd

wb = openpyxl.Workbook()
wb.remove(wb.active)

def write_df(ws, df, start_row=1):
    for j, col in enumerate(df.columns, 1):
        c = ws.cell(row=start_row, column=j, value=col)
        c.font = Font(bold=True, name="Arial")
        c.fill = PatternFill("solid", fgColor="D9E1F2")
    for i, row in enumerate(df.itertuples(index=False), start_row+1):
        for j, val in enumerate(row, 1):
            ws.cell(row=i, column=j, value=val).font = Font(name="Arial")
    return start_row + len(df)

# --- Sheet 1: Contribution Margin ---
df_margin = pd.read_csv("out_margin.csv")
ws1 = wb.create_sheet("Contribution Margin")
last_row = write_df(ws1, df_margin)
chart1 = BarChart(); chart1.title = "Margin % by Dish"; chart1.y_axis.title = "Margin %"
data = Reference(ws1, min_col=5, min_row=1, max_row=last_row)
cats = Reference(ws1, min_col=1, min_row=2, max_row=last_row)
chart1.add_data(data, titles_from_data=True); chart1.set_categories(cats)
ws1.add_chart(chart1, "H2")

# --- Sheet 2: Labor % of Revenue ---
df_labor = pd.read_csv("out_labor.csv")
ws2 = wb.create_sheet("Labor % Revenue")
write_df(ws2, df_labor)

# --- Sheet 3: New Store Performance ---
df_new_store = pd.read_csv("out_new_store.csv")
ws3 = wb.create_sheet("New Store Performance")
last_row3 = write_df(ws3, df_new_store)
chart3 = BarChart(); chart3.title = "Total Revenue by Location"
data3 = Reference(ws3, min_col=5, min_row=1, max_row=last_row3)
cats3 = Reference(ws3, min_col=2, min_row=2, max_row=last_row3)
chart3.add_data(data3, titles_from_data=True); chart3.set_categories(cats3)
ws3.add_chart(chart3, "H2")

# --- Sheet 4: Monthly Revenue Trend (Location 1) ---
df_rev = pd.read_csv("out_monthly_revenue.csv")
df_rev_loc1 = df_rev[df_rev.location_id == 1]
ws4 = wb.create_sheet("Revenue Trend")
last_row4 = write_df(ws4, df_rev_loc1)
chart4 = LineChart(); chart4.title = "Monthly Revenue — Location 1"
data4 = Reference(ws4, min_col=3, min_row=1, max_row=last_row4)
cats4 = Reference(ws4, min_col=2, min_row=2, max_row=last_row4)
chart4.add_data(data4, titles_from_data=True); chart4.set_categories(cats4)
ws4.add_chart(chart4, "H2")

# --- Sheet 5: Forecast ---
df_forecast = pd.read_csv("out_forecast.csv")
ws5 = wb.create_sheet("Revenue Forecast")
write_df(ws5, df_forecast)

# --- Summary sheet with formulas (not hardcoded) ---
ws0 = wb.create_sheet("Summary", 0)
ws0["A1"] = "Doherty Restaurant FP&A Dashboard — Summary"
ws0["A1"].font = Font(bold=True, size=14, name="Arial")
ws0["A3"] = "Avg Contribution Margin %"
ws0["B3"] = "=AVERAGE('Contribution Margin'!E2:E9)"
ws0["A4"] = "Highest Margin Dish"
ws0["B4"] = "=INDEX('Contribution Margin'!A2:A9, MATCH(MAX('Contribution Margin'!E2:E9), 'Contribution Margin'!E2:E9, 0))"
ws0["A5"] = "Total Locations"
ws0["B5"] = f"=COUNTA('New Store Performance'!A2:A{last_row3})"
for cell in ["A3","A4","A5"]:
    ws0[cell].font = Font(bold=True, name="Arial")

wb.save("doherty_fpna_dashboard.xlsx")
print("Dashboard saved: doherty_fpna_dashboard.xlsx")

Dashboard saved: doherty_fpna_dashboard.xlsx
